# Module 7: Collaborative Filtering & Matrix Factorization (SVD)

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Centered-cosine user-user collaborative filtering and SGD latent-factor matrix factorization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM07_Recommender_Systems_SVD.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Centered-Cosine Collaborative Filtering

Subtracting each user's mean rating before taking cosine similarity normalizes harsh vs. generous raters.

In [1]:
import numpy as np


def centered_cosine(R):
    C = np.zeros_like(R, dtype=float)
    for u in range(R.shape[0]):
        m = R[u] > 0
        if np.any(m):
            C[u, m] = R[u, m] - R[u, m].mean()
    nrm = np.linalg.norm(C, axis=1, keepdims=True)
    nrm = np.where(nrm == 0, 1.0, nrm)
    U = C / nrm
    return U @ U.T


R = np.array([
    [5.0, 4.0, 0.0, 1.0, 0.0],
    [4.0, 5.0, 0.0, 1.0, 2.0],
    [1.0, 0.0, 5.0, 4.0, 5.0],
    [0.0, 2.0, 4.0, 5.0, 4.0],
])
sim = centered_cosine(R)
print("User 0 vs User 1 similarity:", round(float(sim[0, 1]), 3))
print("User 0 vs User 2 similarity:", round(float(sim[0, 2]), 3))
assert sim[0, 1] > 0.75 and sim[0, 2] < -0.50


User 0 vs User 1 similarity: 0.824
User 0 vs User 2 similarity: -0.535


## 2. Biased Latent-Factor Matrix Factorization (SVD via SGD)

Factorizing observed ratings as $\hat{r}_{ui} = \mu + b_u + b_i + p_u^\top q_i$ scales linearly in the number of observed ratings $|\Omega|$ rather than filling missing cells with zeros.

In [2]:
obs = [(u, i, float(R[u, i])) for u in range(4) for i in range(5) if R[u, i] > 0]
rng = np.random.default_rng(3003)
mu = float(np.mean([r for _, _, r in obs]))
P = rng.normal(0.0, 0.1, (4, 3))
Q = rng.normal(0.0, 0.1, (5, 3))
bu, bi = np.zeros(4), np.zeros(5)

for _ in range(250):
    for u, i, r in obs:
        err = r - (mu + bu[u] + bi[i] + float(P[u] @ Q[i]))
        bu[u] += 0.03 * (err - 0.05 * bu[u])
        bi[i] += 0.03 * (err - 0.05 * bi[i])
        p_old = P[u].copy()
        P[u] += 0.03 * (err * Q[i] - 0.05 * P[u])
        Q[i] += 0.03 * (err * p_old - 0.05 * Q[i])

rmse = float(np.sqrt(np.mean([(r - (mu + bu[u] + bi[i] + float(P[u] @ Q[i]))) ** 2 for u, i, r in obs])))
u0_i2 = mu + bu[0] + bi[2] + float(P[0] @ Q[2])
print(f"Biased MF RMSE={rmse:.3f} | Predicted rating for (user 0, item 2) = {u0_i2:.2f}")
assert rmse < 0.25


Biased MF RMSE=0.072 | Predicted rating for (user 0, item 2) = 2.87
